In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed
import torch
import torch.nn.functional as F
import numpy as np

In [2]:
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

d:\Documents\anaconda\envs\torch_gpu\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For bett

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [4]:
np.random.seed(1)
print("Number of tokens in dictionary = %d"%(tokenizer.vocab_size))

for i in range(20):
  index = np.random.randint(tokenizer.vocab_size)
  print("Token: %d "%(index)+tokenizer.decode(torch.tensor(index), skip_special_tokens=True))

Number of tokens in dictionary = 50257
Token: 33003  Mormons
Token: 12172  cam
Token: 5192  trig
Token: 32511 ojure
Token: 50057  gist
Token: 43723  Petition
Token: 7813  sin
Token: 21440  Witness
Token: 32912  Remy
Token: 20609 isure
Token: 49100  creeps
Token: 7751  fasc
Token: 43757  Alc
Token: 31228  messenger
Token: 36230  SYSTEM
Token: 32025  precipitation
Token: 21758  cores
Token: 45413  Forestry
Token: 35730  guru
Token: 8444  Disc


In [10]:
def sample_next_token(input_tokens, model, tokenizer):
    # 1. Get model predictions (Logits)
    # Ensure inputs are on the same device as the model (GPU)
    input_ids = input_tokens['input_ids'].to(model.device)
    attn_mask = input_tokens['attention_mask'].to(model.device)
    
    with torch.no_grad(): # Disable gradient calculation for faster inference
        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
    
    # 2. Extract probabilities for the VERY LAST token only
    # outputs.logits shape: [batch, seq_len, vocab_size] -> we take [0, -1, :]
    last_token_logits = outputs.logits[0, -1, :]
    prob_over_tokens = F.softmax(last_token_logits, dim=-1).cpu().numpy()
    
    # 3. Sample the next token index
    # next_token needs to be a simple integer for indexing
    next_token_id = np.random.choice(tokenizer.vocab_size, 1, p=prob_over_tokens)[0]

    # 4. Prepare the new token tensor for concatenation
    # We make it shape (1, 1) to match the (1, seq_len) shape of input_ids
    new_id_tensor = torch.tensor([[next_token_id]], device=model.device)
    new_mask_tensor = torch.tensor([[1]], device=model.device)

    # 5. Append to the dictionary
    input_tokens["input_ids"] = torch.cat((input_ids, new_id_tensor), dim=1)
    input_tokens["attention_mask"] = torch.cat((attn_mask, new_mask_tensor), dim=1)
    
    # Store the probability of the chosen token (optional, for debugging)
    input_tokens['last_token_prob'] = prob_over_tokens[next_token_id]

    return input_tokens

In [14]:
set_seed(0)
input_txt = "Trump"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(40):
  input_tokens = sample_next_token(input_tokens, model, tokenizer)
  print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

Trump only
Trump only dropped
Trump only dropped one
Trump only dropped one statement
Trump only dropped one statement that
Trump only dropped one statement that wasn
Trump only dropped one statement that wasn't
Trump only dropped one statement that wasn't explicitly
Trump only dropped one statement that wasn't explicitly condemning
Trump only dropped one statement that wasn't explicitly condemning Trump
Trump only dropped one statement that wasn't explicitly condemning Trump or
Trump only dropped one statement that wasn't explicitly condemning Trump or associ
Trump only dropped one statement that wasn't explicitly condemning Trump or associating
Trump only dropped one statement that wasn't explicitly condemning Trump or associating himself
Trump only dropped one statement that wasn't explicitly condemning Trump or associating himself with
Trump only dropped one statement that wasn't explicitly condemning Trump or associating himself with the
Trump only dropped one statement that wasn'

In [15]:
def get_best_next_token(input_tokens, model, tokenizer):
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  next_token = [np.argmax(prob_over_tokens)]

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  return output_tokens

In [16]:
# Expected output:
# The best thing about Bath is that it's a place where you can go to
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_best_next_token(input_tokens, model, tokenizer)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

The best thing about Bath is that
The best thing about Bath is that it
The best thing about Bath is that it's
The best thing about Bath is that it's a
The best thing about Bath is that it's a place
The best thing about Bath is that it's a place where
The best thing about Bath is that it's a place where you
The best thing about Bath is that it's a place where you can
The best thing about Bath is that it's a place where you can go
The best thing about Bath is that it's a place where you can go to


In [17]:
def get_top_k_token(input_tokens, model, tokenizer, k=20):
    
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # Draw a sample from the top K most likely tokens.
  # Take copy of the probabilities and sort from largest to smallest (use np.sort)
  # TODO -- replace this line
  sorted_prob_indices = np.argsort(prob_over_tokens)[::-1][:k]

  # Find the probability at the k'th position
  # TODO -- replace this line
  kth_prob_value = prob_over_tokens[sorted_prob_indices[k-1]]

  # Set all probabilities below this value to zero
  prob_over_tokens[prob_over_tokens<kth_prob_value] = 0

  # Renormalize the probabilities so that they sum to one
  # TODO -- replace this line
  prob_over_tokens /= np.sum(prob_over_tokens)

  # Draw random token
  next_token = np.random.choice(len(prob_over_tokens), 1, replace=False, p=prob_over_tokens)
  
  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  return output_tokens

In [18]:
# Expected output:
# The best thing about Bath is that you get to see all the beautiful faces of

set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_top_k_token(input_tokens, model, tokenizer, k=10)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

The best thing about Bath is that
The best thing about Bath is that you
The best thing about Bath is that you get
The best thing about Bath is that you get to
The best thing about Bath is that you get to see
The best thing about Bath is that you get to see all
The best thing about Bath is that you get to see all the
The best thing about Bath is that you get to see all the beautiful
The best thing about Bath is that you get to see all the beautiful faces
The best thing about Bath is that you get to see all the beautiful faces of


In [21]:
def get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh=0.25):
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # Find the most likely tokens that make up the first (thresh) of the probability
  # TODO -- sort the probabilities in decreasing order
  # Replace this line
  sorted_probs_decreasing = np.sort(prob_over_tokens)[::-1]
    
  # TODO -- compute the cumulative sum of these probabilities
  # Replace this line
  cum_sum_probs = np.cumsum(sorted_probs_decreasing)

  # Find index where that the cumulative sum is greater than the threshold
  thresh_index = np.argmax(cum_sum_probs>thresh)

  print("Choosing from %d tokens"%(thresh_index))
  # TODO:  Find the probability value to threshold
  # Replace this line:
  thresh_prob = sorted_probs_decreasing[thresh_index] #0.5

  # Set any probabilities less than this to zero
  prob_over_tokens[prob_over_tokens<thresh_prob] = 0
  # Renormalize
  prob_over_tokens = prob_over_tokens / np.sum(prob_over_tokens)
  # Draw random token
  next_token = np.random.choice(len(prob_over_tokens), 1, replace=False, p=prob_over_tokens)

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]

In [22]:
# Expected output:
# The best thing about Bath is that it's not a city that has been around
set_seed(0)
input_txt = "The best thing about Bath is"
input_tokens = tokenizer(input_txt, return_tensors='pt')
for i in range(10):
    input_tokens = get_nucleus_sampling_token(input_tokens, model, tokenizer, thresh = 0.2)
    print(tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True))

     

Choosing from 0 tokens


TypeError: 'NoneType' object is not subscriptable

In [23]:
# This routine returns the k'th most likely next token.
# If k =0 then it returns the most likely token, if k=1 it returns the next most likely and so on
# We will need this for beam search
def get_kth_most_likely_token(input_tokens, model, tokenizer, k):
    
  # Run model to get prediction over next output
  outputs = model(input_ids = input_tokens['input_ids'], attention_mask = input_tokens['attention_mask'])
  # Find prediction
  prob_over_tokens = F.softmax(outputs.logits, dim=-1).detach().numpy()[0,-1]

  # Find the k'th most likely token
  # TODO Sort the probabilities from largest to smallest
  # Replace this line:
  sorted_prob_over_tokens = np.sort(prob_over_tokens)[::-1]
  # TODO Find the k'th sorted probability
  # Replace this line
  kth_prob_value = sorted_prob_over_tokens[k-1]

  # Find position of this token.
  next_token = np.where(prob_over_tokens == kth_prob_value)[0]

  # Append token to sentence
  output_tokens = input_tokens
  output_tokens["input_ids"] = torch.cat((output_tokens['input_ids'],torch.tensor([next_token])),dim=1)
  output_tokens['attention_mask'] = torch.cat((output_tokens['attention_mask'],torch.tensor([[1]])),dim=1)
  output_tokens['last_token_prob'] = prob_over_tokens[next_token]
  output_tokens['log_prob'] = output_tokens['log_prob'] + np.log(prob_over_tokens[next_token])
  return output_tokens